In [11]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

In [2]:
# Loading the pre-split data from preprocessing
train_df = pd.read_csv('../data/train.csv')
val_df = pd.read_csv('../data/val.csv')
test_df = pd.read_csv('../data/test.csv')

print(f"Train: {train_df.shape}")
print(f"Val: {val_df.shape}")
print(f"Test: {test_df.shape}")

Train: (139666, 42)
Val: (29928, 42)
Test: (29929, 42)


In [ ]:
# Helper to apply same mapping to all three splits at once

def apply_map(col, mapping):
    train_df[col] = train_df[col].map(mapping)
    val_df[col] = val_df[col].map(mapping)
    test_df[col] = test_df[col].map(mapping)

In [6]:
train_df.groupby('hispanic origin')['label'].mean().sort_values(ascending=False)

hispanic origin
All other                    0.068560
Cuban                        0.059949
Other Spanish                0.031144
Do not know                  0.023148
Central or South American    0.021529
Mexican-American             0.020896
Puerto Rican                 0.018655
Chicano                      0.018605
Mexican (Mexicano)           0.010176
Name: label, dtype: float64

In [9]:
train_df.groupby('country of birth self')['label'].mean().sort_values(ascending=False)#.head(20)

country of birth self
Iran                            0.186916
Scotland                        0.183333
Taiwan                          0.179856
India                           0.178082
Hungary                         0.133333
France                          0.129870
Japan                           0.125000
England                         0.122186
Ireland                         0.117647
Greece                          0.114583
Canada                          0.108055
Yugoslavia                      0.100000
Thailand                        0.095238
Philippines                     0.091973
?                               0.088581
Germany                         0.083612
Hong Kong                       0.080645
Cuba                            0.077320
China                           0.074627
Holand-Netherlands              0.071429
Italy                           0.066901
Poland                          0.064151
United-States                   0.063000
Portugal                        0.0

In [10]:
print(train_df['country of birth self'].value_counts())

country of birth self
United-States                   123825
Mexico                            4078
?                                 2382
Puerto-Rico                        987
Philippines                        598
Germany                            598
Cuba                               582
Canada                             509
El-Salvador                        499
Dominican-Republic                 479
China                              335
South Korea                        331
Columbia                           314
England                            311
India                              292
Italy                              284
Vietnam                            279
Poland                             265
Guatemala                          252
Japan                              224
Jamaica                            210
Ecuador                            196
Peru                               194
Nicaragua                          153
Haiti                              149
Tai

In [18]:
# Dropping following columns

# Numeric codes (detailed industry/occupation recode): captured by major occupation code
# (wage per hour): majority 0, misleading
# Near zero variance (enroll in edu inst, reason for unemployment): 98%+ single category
# Geographic columns (region/state of previous residence): weak signal, potential bias
# Migration columns: too many missing values marked as '?'
# Hispanic origin dropped: weak signal, captured by citizenship

cols_to_drop = [
    'detailed industry recode',
    'detailed occupation recode',
    'major industry code',
    'wage per hour',
    'enroll in edu inst last wk',
    'reason for unemployment',
    'member of a labor union',
    'region of previous residence',
    'state of previous residence',
    'detailed household and family stat',
    'migration code-change in msa',
    'migration code-change in reg',
    'migration code-move within reg',
    'live in this house 1 year ago',
    'migration prev res in sunbelt',
    'family members under 18',
    'country of birth father',
    'country of birth mother',
    'country of birth self',
    'year',
    'hispanic origin'
]

train_df.drop(columns=cols_to_drop, inplace=True)
val_df.drop(columns=cols_to_drop, inplace=True)
test_df.drop(columns=cols_to_drop, inplace=True)

print(f"Remaining columns: {train_df.columns.tolist()}")
print(f"Shape after dropping: {train_df.shape}")

Remaining columns: ['age', 'class of worker', 'education', 'marital stat', 'major occupation code', 'race', 'sex', 'full or part time employment stat', 'capital gains', 'capital losses', 'dividends from stocks', 'tax filer stat', 'detailed household summary in household', 'weight', 'num persons worked for employer', 'citizenship', 'own business or self employed', "fill inc questionnaire for veteran's admin", 'veterans benefits', 'weeks worked in year', 'label']
Shape after dropping: (139666, 21)


In [13]:
train_df.groupby((train_df['wage per hour'] > 0))['label'].mean()

wage per hour
False    0.062748
True     0.050579
Name: label, dtype: float64

In [17]:
print(train_df.groupby('region of previous residence')['label'].mean().sort_values(ascending=False))
a = train_df.groupby('state of previous residence')['label'].mean().sort_values(ascending=False)
a.head()

region of previous residence
Not in universe    0.064044
Northeast          0.057641
West               0.038678
South              0.036918
Abroad             0.036364
Midwest            0.028197
Name: label, dtype: float64


state of previous residence
District of Columbia    0.123457
Idaho                   0.117647
Connecticut             0.105882
New Jersey              0.081633
New Mexico              0.072727
Name: label, dtype: float64

Binary Features

In [19]:
# Capital gains, losses, dividends are zero-inflated (95%+ zeros)
# Having any investment activity is a strong wealth signal regardless of amount

for col in ['capital gains', 'capital losses', 'dividends from stocks']:
    train_df[col] = (train_df[col] > 0).astype(int)
    val_df[col] = (val_df[col] > 0).astype(int)
    test_df[col] = (test_df[col] > 0).astype(int)

Sex

In [20]:
sex_map = {'Male': 1, 'Female': 0}
apply_map('sex', sex_map)

Race

In [21]:
# Race shows income disparity across groups
# Ordinal encoding based on positive income rate
race_map = {
    'Amer Indian Aleut or Eskimo': 0,
    'Other': 0,
    'Black': 1,
    'White': 2,
    'Asian or Pacific Islander': 2
}
apply_map('race', race_map)

Education

In [22]:
# Three tier grouping based on income rate analysis on train set
# Tier 0: below high school (limited earning potential)
# Tier 1: high school/some college (middle income)
# Tier 2: bachelor's and above (high earning potential)
education_map = {
    'Less than 1st grade': 0,
    '1st 2nd 3rd or 4th grade': 0,
    '5th or 6th grade': 0,
    '7th and 8th grade': 0,
    '9th grade': 0,
    '10th grade': 0,
    '11th grade': 0,
    '12th grade no diploma': 0,
    'Children': 0,
    'High school graduate': 1,
    'Some college but no degree': 1,
    'Associates degree-occup /vocational': 1,
    'Associates degree-academic program': 1,
    'Bachelors degree(BA AB BS)': 2,
    'Masters degree(MA MS MEng MEd MSW MBA)': 2,
    'Prof school degree (MD DDS DVM LLB JD)': 2,
    'Doctorate degree(PhD EdD)': 2
}
apply_map('education', education_map)

Marital Status

In [23]:
# Married couples benefit from dual income and shared expenses
# Grouping by financial stability and income potential
marital_map = {
    'Never married': 0,
    'Widowed': 0,
    'Divorced': 0,
    'Separated': 1,
    'Married-spouse absent': 1,
    'Married-A F spouse present': 2,
    'Married-civilian spouse present': 2
}
apply_map('marital stat', marital_map)

Worker Class

In [24]:
# Ranking employment type by income potential
# Private sector and incorporated self-employed tend to earn most
worker_map = {
    'Never worked': 0,
    'Without pay': 0,
    'State government': 1,
    'Federal government': 1,
    'Local government': 1,
    'Not in universe': 1,
    'Self-employed-not incorporated': 2,
    'Self-employed-incorporated': 2,
    'Private': 3
}
apply_map('class of worker', worker_map)

In [25]:
# Grouping occupations by income potential based on train set rate analysis
# Executive and professional roles dominate high earners
occupation_map = {
    'Not in universe': 1,
    'Private household services': 0,
    'Handlers equip cleaners etc': 0,
    'Farming forestry and fishing': 0,
    'Other service': 0,
    'Armed Forces': 0,
    'Machine operators assmblrs & inspctrs': 0,
    'Protective services': 0,
    'Transportation and material moving': 0,
    'Technicians and related support': 0,
    'Adm support including clerical': 0,
    'Precision production craft & repair': 1,
    'Sales': 1,
    'Professional specialty': 2,
    'Executive admin and managerial': 2
}
apply_map('major occupation code', occupation_map)

In [32]:
# Grouping by employment intensity and income rate analysis
employment_map = {
    'Full-time schedules': 2,
    'PT for econ reasons usually PT': 2,
    'PT for non-econ reasons usually FT': 2,
    'PT for econ reasons usually FT': 1,
    'Children or Armed Forces': 1,
    'Unemployed full-time': 1,
    'Unemployed part- time': 0,
    'Not in labor force': 0
}
apply_map('full or part time employment stat', employment_map)

In [26]:
# Joint filers represent dual income households - strongest predictor
# Nonfilers are almost certainly low income (0.05% positive rate)
tax_map = {
    'Nonfiler': 0,
    'Single': 1,
    'Head of household': 1,
    'Joint both 65+': 1,
    'Joint one under 65 & one 65+': 2,
    'Joint both under 65': 2
}
apply_map('tax filer stat', tax_map)

Household data

In [27]:
# Householders and spouses are primary earners in a household
household_map = {
    'Householder': 1,
    'Spouse of householder': 1,
    'Nonrelative of householder': 1,
    'Other relative of householder': 0,
    'Child 18 or older': 0,
    'Group Quarters- Secondary individual': 0,
    'Child under 18 never married': 0,
    'Child under 18 ever married': 0
}
apply_map('detailed household summary in household', household_map)

In [29]:
# Naturalized citizens show highest income rate - tend to be skilled immigrants
# Non-citizens show lowest income rate
citizenship_map = {
    'Foreign born- U S citizen by naturalization': 2,
    'Native- Born in the United States': 1,
    'Native- Born abroad of American Parent(s)': 1,
    'Foreign born- Not a citizen of U S': 0,
    'Native- Born in Puerto Rico or U S Outlying': 0
}
apply_map('citizenship', citizenship_map)

Other features

In [30]:
# Own business: higher value = more entrepreneurial = higher income potential
own_business_map = {0: 0, 2: 1, 1: 2}
apply_map('own business or self employed', own_business_map)

# Veterans admin questionnaire: No has higher income rate than Yes
veteran_map = {'Yes': 0, 'Not in universe': 1, 'No': 2}
apply_map("fill inc questionnaire for veteran's admin", veteran_map)

# Veterans benefits: binary - having any benefit is a signal
train_df['veterans benefits'] = (train_df['veterans benefits'] > 0).astype(int)
val_df['veterans benefits'] = (val_df['veterans benefits'] > 0).astype(int)
test_df['veterans benefits'] = (test_df['veterans benefits'] > 0).astype(int)

# Employment indicator: binary - working vs not working
train_df['num persons worked for employer'] = (train_df['num persons worked for employer'] > 0).astype(int)
val_df['num persons worked for employer'] = (val_df['num persons worked for employer'] > 0).astype(int)
test_df['num persons worked for employer'] = (test_df['num persons worked for employer'] > 0).astype(int)

Handle Nan values

In [33]:
# Filling with 0 as unseen categories are likely low income groups
print("NaN counts before fill:")
print(train_df.isnull().sum()[train_df.isnull().sum() > 0])

train_df.fillna(0, inplace=True)
val_df.fillna(0, inplace=True)
test_df.fillna(0, inplace=True)

print(f"\nNaN counts after fill: {train_df.isnull().sum().sum()}")

NaN counts before fill:
major occupation code                  2891
full or part time employment stat    139666
citizenship                          139666
dtype: int64

NaN counts after fill: 0


Normalizing continuous features

In [35]:
# Normalizing age and weeks worked for Logistic Regression compatibility

scaler = MinMaxScaler()
train_df[['age', 'weeks worked in year']] = scaler.fit_transform(train_df[['age', 'weeks worked in year']]) # fit only on train, transform on val and test to prevent data leakage
val_df[['age', 'weeks worked in year']] = scaler.transform(val_df[['age', 'weeks worked in year']])
test_df[['age', 'weeks worked in year']] = scaler.transform(test_df[['age', 'weeks worked in year']])

Save 'Weight' column

In [36]:
# Will use 'weight' column as sample_weight during model training to account for population representation

# Extracting weight column separately - not a feature
# Used as sample_weight during model training to account for population representation
# Normalizing weights so they sum to 1 - represents population proportions
total_weight = pd.concat([train_df['weight'], val_df['weight'], test_df['weight']]).sum()

w_train = train_df['weight'] / total_weight
w_val = val_df['weight'] / total_weight
w_test = test_df['weight'] / total_weight

train_df.drop(columns=['weight'], inplace=True)
val_df.drop(columns=['weight'], inplace=True)
test_df.drop(columns=['weight'], inplace=True)

w_train.to_csv('../data/w_train.csv', index=False)
w_val.to_csv('../data/w_val.csv', index=False)
w_test.to_csv('../data/w_test.csv', index=False)

print("Weights saved!")

Weights saved!


Save proessed data

In [37]:
# Saving processed splits for training notebook
train_df.to_csv('../data/train_processed.csv', index=False)
val_df.to_csv('../data/val_processed.csv', index=False)
test_df.to_csv('../data/test_processed.csv', index=False)

print(f"Final feature set: {train_df.columns.tolist()}")
print(f"Train shape: {train_df.shape}")
print(f"Val shape: {val_df.shape}")
print(f"Test shape: {test_df.shape}")

Final feature set: ['age', 'class of worker', 'education', 'marital stat', 'major occupation code', 'race', 'sex', 'full or part time employment stat', 'capital gains', 'capital losses', 'dividends from stocks', 'tax filer stat', 'detailed household summary in household', 'num persons worked for employer', 'citizenship', 'own business or self employed', "fill inc questionnaire for veteran's admin", 'veterans benefits', 'weeks worked in year', 'label']
Train shape: (139666, 20)
Val shape: (29928, 20)
Test shape: (29929, 20)
